# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jerovernay/FlyRank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The rule, in plain words:** a page is worth flagging for a snippet/CTR fix if it gets
real search visibility this month (impressions ≥ 100, a volume floor) and its click-through
rate sits below the median CTR of pages ranked in the same position tier. If similar-ranked
pages get meaningfully more clicks per impression, the position isn't the problem — the
snippet (title/meta) is. This is the CTR-fix logic from the Week 4 session, applied literally.

**Signal check 1 — CTR vs. position (behind the CTR-fix flag).** Hypothesis: CTR should
fall as position gets worse, so "underperforming CTR for your tier" is a meaningful,
non-arbitrary benchmark (not one global threshold for every page regardless of rank).

**Signal check 2 — staleness (behind the refresh flag).** Hypothesis: content not touched
in a long time (`content_updated_date` far in the past) should show weaker performance,
supporting a staleness-based refresh flag.

Both checks below use the same volume floor (`total_impressions ≥ 100` over `month=2026-03`)
so low-traffic noise doesn't distort the bucket means — same caveat as the `position_tier`
median warning in the data dictionary.

**Result, signal 1 (CTR vs. position) — verdict: CONFIRMED.** Median CTR falls
monotonically from `top_3` (0.211%) → `page_1` (0.190%) → `striking` (0.101%) →
`page_3_5`/`deep` (0.0%), n = 10,189 / 47,787 / 19,545 / 19,757 / 4,131. Position tier is a
real, ordered benchmark — comparing a page's CTR to its own tier's median is honest, not
arbitrary.

**Result, signal 2 (staleness) — verdict: FALSE.** `content_updated_date` comes from
`dim_content`, which is a *live, present-day* dimension table, not a March-specific
snapshot — so "days since update" measured against the March window is mostly negative:
82.2% of volume-floor rows (83,330 of 101,409) show an update date *after* March 2026 ended.
Bucketed by days-since-update relative to month-end: `updated_after_month` n=83,330 (mean
CTR 0.275%), `<90` n=17,941 (0.198%), `90-180` n=123 (0.316%), `181-365` n=15 (0.294%) — no
usable staleness signal, because the column doesn't describe the state of the page *during*
the scored month. Using it as-is would have quietly leaked a future, live-table timestamp
into a rule meant to score March behavior. **The rule below drops staleness entirely** and
uses only the confirmed CTR-vs-position signal — a negative result that saved the rule from
a broken input, not a wasted check.

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np

# HF_TOKEN lives in a local, gitignored .env — never paste a token into a cell (public repo!)
with open("../../.env") as f:
    for line in f:
        if line.startswith("HF_TOKEN"):
            os.environ["HF_TOKEN"] = line.strip().split("=", 1)[1]

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

MONTH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
DIM_CONTENT = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"
SNAPSHOT = pd.Timestamp("2026-03-31")  # last day of the scored month
VOL_FLOOR = 100  # impressions floor, same reasoning as the position_tier volume-floor caveat

# One row = one content item, aggregated over month=2026-03 (same grain as ML-04/ML-05)
base = con.sql(f"""
    WITH gsc_agg AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS total_impressions,
               SUM(gsc_clicks) AS total_clicks,
               SUM(gsc_impressions * gsc_avg_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
        FROM read_parquet('{MONTH}')
        GROUP BY content_hash_id
    )
    SELECT g.*, dc.content_updated_date, dc.is_published, dc.is_deleted
    FROM gsc_agg g
    LEFT JOIN read_parquet('{DIM_CONTENT}') dc USING (content_hash_id)
""").df()

df = base[(base.is_published) & (~base.is_deleted)].copy()
print("rows after publish/delete filter:", len(df))

# --- Signal 1: CTR vs. position tier (behind the CTR-fix flag) ---
scored = df[(df.total_impressions > 0) & (df.avg_position > 0)].copy()
scored["ctr"] = scored.total_clicks / scored.total_impressions * 100
vol = scored[scored.total_impressions >= VOL_FLOOR].copy()

pos_bins = [-1, 3, 10, 20, 50, 10_000]
pos_labels = ["top_3", "page_1", "striking", "page_3_5", "deep"]
vol["pos_tier"] = pd.cut(vol["avg_position"], bins=pos_bins, labels=pos_labels)

pos_table = vol.groupby("pos_tier", observed=True)["ctr"].agg(n="count", median_ctr="median", mean_ctr="mean")
print("\nSignal 1 — CTR by position tier (volume >= 100 impressions):")
print(pos_table.round(3))
print("Verdict: CONFIRMED — median CTR falls monotonically as position worsens.")

# --- Signal 2: staleness vs. performance (behind the refresh flag) ---
stale = df.copy()
stale["days_since_update"] = (SNAPSHOT - pd.to_datetime(stale["content_updated_date"])).dt.days
stale["ctr"] = np.where(stale.total_impressions > 0, stale.total_clicks / stale.total_impressions * 100, np.nan)
stale_vol = stale[stale.total_impressions >= VOL_FLOOR].copy()

pct_after_month = (stale_vol["days_since_update"] < 0).mean()
stale_bins = [-100_000, 0, 90, 180, 365, 100_000]
stale_labels = ["updated_after_month", "<90", "90-180", "181-365", "365+"]
stale_vol["stale_tier"] = pd.cut(stale_vol["days_since_update"], bins=stale_bins, labels=stale_labels)

stale_table = stale_vol.groupby("stale_tier", observed=True)["ctr"].agg(n="count", mean_ctr="mean")
print(f"\nSignal 2 — CTR by staleness tier (volume >= 100 impressions):")
print(f"share of rows with content_updated_date AFTER month-end (impossible for a real")
print(f"in-month staleness signal): {pct_after_month:.1%}")
print(stale_table.round(3))
print("Verdict: FALSE — content_updated_date is a live/present-day field, not a")
print("March-specific snapshot, so it cannot measure staleness during the scored month.")


rows after publish/delete filter: 321106

Signal 1 — CTR by position tier (volume >= 100 impressions):
              n  median_ctr  mean_ctr
pos_tier                             
top_3     10189       0.211     0.340
page_1    47787       0.190     0.320
striking  19545       0.101     0.244
page_3_5  19757       0.000     0.143
deep       4131       0.000     0.046
Verdict: CONFIRMED — median CTR falls monotonically as position worsens.

Signal 2 — CTR by staleness tier (volume >= 100 impressions):
share of rows with content_updated_date AFTER month-end (impossible for a real
in-month staleness signal): 82.2%
                         n  mean_ctr
stale_tier                          
updated_after_month  83330     0.275
<90                  17941     0.198
90-180                 123     0.316
181-365                 15     0.294
Verdict: FALSE — content_updated_date is a live/present-day field, not a
March-specific snapshot, so it cannot measure staleness during the scored month.


## 2. Build the ranked queue (writes the CSV)

Score = `total_impressions × (peer_median_ctr − ctr) / 100` — the estimated clicks/month a
page is leaving on the table if it caught up to the median CTR of its own position tier.
Only pages that clear the volume floor **and** sit below their tier's median CTR get a
non-zero score; the reason code and action are ONE pair, fired by this one rule:

- `reason_code = "ctr_below_position_peers"`, `action = "snippet_fix"` — score > 0
- `reason_code = "none"`, `action = "monitor"` — everything else (below floor, at/above
  peer median, or missing position data)

No fitted weights, no product flags, no future window — every input (`total_impressions`,
`total_clicks`, `avg_position`) is aggregated from `month=2026-03` GSC rows only, the same
month being scored.

In [2]:
# Peer benchmark: median CTR per position tier, computed on the SAME volume-floor slice
# used to confirm signal 1 above — no leakage of a different data cut into the rule.
peer_median_ctr = vol.groupby("pos_tier", observed=True)["ctr"].median()

queue = scored.copy()  # impressions > 0 and avg_position > 0, no volume filter yet
queue["pos_tier"] = pd.cut(queue["avg_position"], bins=pos_bins, labels=pos_labels)
queue["peer_median_ctr"] = queue["pos_tier"].map(peer_median_ctr)
queue["ctr_gap_pp"] = queue["peer_median_ctr"] - queue["ctr"]
queue["volume_ok"] = queue["total_impressions"] >= VOL_FLOOR

flagged = queue["volume_ok"] & (queue["ctr_gap_pp"] > 0)
queue["score"] = np.where(flagged, queue["total_impressions"] * queue["ctr_gap_pp"] / 100, 0.0).round(2)
queue["reason_code"] = np.where(flagged, "ctr_below_position_peers", "none")
queue["action"] = np.where(flagged, "snippet_fix", "monitor")

queue = queue.sort_values("score", ascending=False).reset_index(drop=True)
print("total rows:", len(queue), "| flagged (snippet_fix):", flagged.sum(), "| monitor:", (~flagged).sum())

out_cols = ["content_hash_id", "pos_tier", "avg_position", "total_impressions", "total_clicks",
            "ctr", "peer_median_ctr", "ctr_gap_pp", "score", "reason_code", "action"]
os.makedirs("../outputs", exist_ok=True)
queue[out_cols].to_csv("../outputs/baseline_action_score.csv", index=False)
print("wrote", len(queue), "rows to work/outputs/baseline_action_score.csv")
queue[out_cols].head(10)


total rows: 175136 | flagged (snippet_fix): 38745 | monitor: 136391


wrote 175136 rows to work/outputs/baseline_action_score.csv


,content_hash_id,pos_tier,avg_position,total_impressions,total_clicks,ctr,peer_median_ctr,ctr_gap_pp,score,reason_code,action
0,content_44f34c0a90047651,top_3,0.665877,212404.0,24.0,0.011299,0.211416,0.200117,425.06,ctr_below_position_peers,snippet_fix
1,content_8e1334d6356668e3,top_3,2.693038,134984.0,1.0,0.000741,0.211416,0.210676,284.38,ctr_below_position_peers,snippet_fix
2,content_fec55986a1868d62,top_3,0.308426,124075.0,1.0,0.000806,0.211416,0.210611,261.32,ctr_below_position_peers,snippet_fix
3,content_34a70fea29d15f24,page_1,3.166132,143019.0,43.0,0.030066,0.190114,0.160048,228.90,ctr_below_position_peers,snippet_fix
4,content_f6116743b00afc2d,page_1,9.735658,107584.0,15.0,0.013943,0.190114,0.176171,189.53,ctr_below_position_peers,snippet_fix
5,content_9c057b66c30a3abb,top_3,0.116003,83834.0,1.0,0.001193,0.211416,0.210224,176.24,ctr_below_position_peers,snippet_fix
6,content_7c6373141eae744a,page_1,5.948459,132593.0,83.0,0.062598,0.190114,0.127516,169.08,ctr_below_position_peers,snippet_fix
7,content_cd3d932d4e1c8db0,page_1,7.831807,89332.0,4.0,0.004478,0.190114,0.185636,165.83,ctr_below_position_peers,snippet_fix
8,content_046fc480045b88f5,page_1,7.208276,83788.0,6.0,0.007161,0.190114,0.182953,153.29,ctr_below_position_peers,snippet_fix
9,content_9540d884af3e41fd,page_1,8.005184,82376.0,11.0,0.013353,0.190114,0.176761,145.61,ctr_below_position_peers,snippet_fix


## 3. Top-10 review

Reading the top of the queue by hand (per the `building-baselines` skill: "the top 20 is
where bad logic shows itself" — here, the top 10). All ten are `top_3`/`page_1` pages with
six-figure impressions and near-zero clicks — the pattern the rule is built to catch. One
line each: the action, why it's there, what would make it wrong.

1. `content_44f34c0a90047651` — snippet_fix. avg_position 0.67 (top_3), 212,404 impressions,
   CTR 0.011% vs. tier median 0.211%. **Wrong if:** the sub-1.0 average position is a data
   artifact (weighted daily averages below 1 shouldn't be possible for real GSC ranks) —
   worth a raw-row check before treating this as a real snippet problem.
2. `content_8e1334d6356668e3` — snippet_fix. position 2.69, 134,984 impressions, 1 total
   click. **Wrong if:** this is a duplicate/canonical URL cannibalizing clicks that land on
   a sibling page instead — the fix would be a canonical/merge, not a snippet rewrite.
3. `content_fec55986a1868d62` — snippet_fix. position 0.31, 124,075 impressions, 1 click.
   **Wrong if:** the query is branded/navigational and users are already satisfied by the
   snippet text alone (e.g. an FAQ rich result answers the query without a click).
4. `content_34a70fea29d15f24` — snippet_fix. position 3.17 (page_1), 143,019 impressions,
   43 clicks, CTR 0.030% vs. median 0.190%. **Wrong if:** a competing page (ad, featured
   snippet, "People also ask") is absorbing clicks on this specific query — not fixable by
   editing this page's own title/meta.
5. `content_f6116743b00afc2d` — snippet_fix. position 9.74, 107,584 impressions, 15 clicks.
   **Wrong if:** the underlying query mix is highly seasonal and this month caught an
   unusually high-impression, low-intent spike (e.g. a trend article that briefly ranked).
6. `content_9c057b66c30a3abb` — snippet_fix. position 0.12, 83,834 impressions, 1 click.
   **Wrong if:** same as #1 — a near-zero average position with this little click activity
   looks more like a tracking/attribution glitch than a genuine snippet problem.
7. `content_7c6373141eae744a` — snippet_fix. position 5.95, 132,593 impressions, 83 clicks.
   **Wrong if:** the client recently changed the title/meta already and this month's data
   predates the fix taking effect — the flag would be stale by the time anyone acts on it.
8. `content_cd3d932d4e1c8db0` — snippet_fix. position 7.83, 89,332 impressions, 4 clicks.
   **Wrong if:** the page ranks for a broad/generic keyword where most searchers' intent
   doesn't match the page at all (an intent mismatch, not a snippet issue).
9. `content_046fc480045b88f5` — snippet_fix. position 7.21, 83,788 impressions, 6 clicks.
   **Wrong if:** this client's GSC property has partial tracking (e.g. AMP or app-indexed
   traffic reported separately), understating real clicks for this specific page.
10. `content_9540d884af3e41fd` — snippet_fix. position 8.01, 82,376 impressions, 11 clicks.
    **Wrong if:** the page is genuinely well-served by its current snippet and the low CTR
    reflects low commercial intent for this exact query, not a fixable presentation issue.

In [3]:
# Reproduce the top-10 rows the review above is based on
queue[out_cols].head(10)


,content_hash_id,pos_tier,avg_position,total_impressions,total_clicks,ctr,peer_median_ctr,ctr_gap_pp,score,reason_code,action
0,content_44f34c0a90047651,top_3,0.665877,212404.0,24.0,0.011299,0.211416,0.200117,425.06,ctr_below_position_peers,snippet_fix
1,content_8e1334d6356668e3,top_3,2.693038,134984.0,1.0,0.000741,0.211416,0.210676,284.38,ctr_below_position_peers,snippet_fix
2,content_fec55986a1868d62,top_3,0.308426,124075.0,1.0,0.000806,0.211416,0.210611,261.32,ctr_below_position_peers,snippet_fix
3,content_34a70fea29d15f24,page_1,3.166132,143019.0,43.0,0.030066,0.190114,0.160048,228.90,ctr_below_position_peers,snippet_fix
4,content_f6116743b00afc2d,page_1,9.735658,107584.0,15.0,0.013943,0.190114,0.176171,189.53,ctr_below_position_peers,snippet_fix
5,content_9c057b66c30a3abb,top_3,0.116003,83834.0,1.0,0.001193,0.211416,0.210224,176.24,ctr_below_position_peers,snippet_fix
6,content_7c6373141eae744a,page_1,5.948459,132593.0,83.0,0.062598,0.190114,0.127516,169.08,ctr_below_position_peers,snippet_fix
7,content_cd3d932d4e1c8db0,page_1,7.831807,89332.0,4.0,0.004478,0.190114,0.185636,165.83,ctr_below_position_peers,snippet_fix
8,content_046fc480045b88f5,page_1,7.208276,83788.0,6.0,0.007161,0.190114,0.182953,153.29,ctr_below_position_peers,snippet_fix
9,content_9540d884af3e41fd,page_1,8.005184,82376.0,11.0,0.013353,0.190114,0.176761,145.61,ctr_below_position_peers,snippet_fix


## 4. Weak picks + leakage check

**Weakest picks in the top 10:** #1 (`content_44f34c0a90047651`) and #6
(`content_9c057b66c30a3abb`) both carry an average position **below 1.0** with almost no
clicks despite six-figure impressions — that combination reads more like a GSC weighted-
average artifact or an attribution glitch than a real, fixable snippet problem. #7
(`content_7c6373141eae744a`) is weak for a different reason: it already has a somewhat
healthy click count (83), so a "recent fix already applied, this month is stale data" story
is plausible — the rule can't distinguish "still broken" from "just fixed."

**Leakage check.** The rule uses exactly three inputs, all aggregated from
`month=2026-03` GSC rows only — `total_impressions`, `total_clicks`, `avg_position` — plus
the position-tier bucketing derived from `avg_position` itself. No column from a later
month, no `trend_pct`/`trend_direction`-style derived label, no product/CRM flag. The one
column considered and explicitly rejected was `content_updated_date`
(Section 1, signal 2) — it comes from the live `dim_content` table, not a March snapshot,
and using it would have pulled a future-dated timestamp into a rule scoring March
performance. It never made it into `queue` or the CSV.

In [4]:
# Confirm the rejected signal never leaked into the modeling inputs
rule_inputs = ["total_impressions", "total_clicks", "avg_position"]
excluded = ["content_updated_date", "trend_pct", "trend_direction"]
leaked_in = [c for c in excluded if c in rule_inputs or c in queue[out_cols].columns]
print("excluded/rejected fields present in the rule's inputs or output CSV (should be empty):", leaked_in)
print("rule inputs used:", rule_inputs)

# Sub-1.0 avg_position rows flagged in the review — how common are they across the whole queue?
below_one = (queue["avg_position"] < 1.0).sum()
print(f"\nrows with avg_position < 1.0 across the full scored queue: {below_one} of {len(queue)}"
      f" ({below_one/len(queue):.2%}) — a data quirk worth a raw-row follow-up, not disqualifying")


excluded/rejected fields present in the rule's inputs or output CSV (should be empty): []
rule inputs used: ['total_impressions', 'total_clicks', 'avg_position']

rows with avg_position < 1.0 across the full scored queue: 1301 of 175136 (0.74%) — a data quirk worth a raw-row follow-up, not disqualifying


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.